# Deep Learning: Next-Session Browsing Category & Profile Forecasting

This notebook builds, trains, and evaluates a **Multi-Task PyTorch LSTM Neural Network** that uses the previous **5 browsing sessions** to predict:
1. **Next Session Dominant Category** (Classification)
2. **Next Session Quantitative Profile** (Duration, RAM usage, Event count regression)

All experiment parameters, training loss curves, evaluation metrics, and PyTorch model checkpoints are logged directly to **MLflow**.

## 1. Import Dependencies and Setup MLflow

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from src.utils.mlflow_utils import setup_mlflow, get_tracking_uri
from src.modeling.dataset_builder import DatasetBuilder
from src.modeling.lstm_pipeline import LSTMPipeline

# Initialize MLflow tracking backend
setup_mlflow()
print(f"MLflow Tracking URI: {get_tracking_uri()}")

## 2. Build Gold Sequence Datasets

Using a sliding window of sequence length = 5 sessions.

In [ ]:
builder = DatasetBuilder(sequence_length=5)
gold_data = builder.run()
print("Gold Dataset Keys:", list(gold_data.keys()))
print(f"Sequence dataset contains {len(gold_data['sequence_metadata'])} window samples.")

## 3. Execute PyTorch LSTM Training & Evaluation Pipeline

In [ ]:
pipeline = LSTMPipeline()
results = pipeline.run()

print("Pipeline Training Summary:")
print(f"  - MLflow Run ID: {results.get('run_id')}")
print(f"  - Test Category RMSE: {results['metrics']['test_rmse']:.4f}")

## 4. Visualize Training & Validation Loss Curves

In [ ]:
history = results.get("history", [])
if history:
    epochs = [entry["epoch"] for entry in history]
    val_loss = [entry["validation_loss"] for entry in history]
    
    plt.figure(figsize=(9, 5))
    plt.plot(epochs, val_loss, label="Validation Loss", marker="o", color="#1f77b4")
    plt.title("PyTorch Multi-Task LSTM Training History", fontsize=14, fontweight="bold")
    plt.xlabel("Epoch", fontsize=11)
    plt.ylabel("Loss", fontsize=11)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## 5. Next-Session Forecast Predictions & Accuracy Inspection

In [ ]:
predictions_df = results["predictions"]
print(f"Total Test Predictions Evaluated: {len(predictions_df)}")
display(predictions_df.head(10))